# blayers `fit()` API demo

This notebook shows how to use the high-level `fit()` helper instead of wiring up SVI manually.
The same model runs unchanged under VI, MCMC, and SVGD — just swap `method=`.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as random
import matplotlib.pyplot as plt

from blayers.fit import fit
from blayers.layers import AdaptiveLayer, InterceptLayer
from blayers.links import gaussian_link
from blayers.sampling import autoreshape

## 1. Data

In [ ]:
key = random.PRNGKey(0)
n, d = 1024, 6

key, kx, kb, kn = random.split(key, 4)
X = random.normal(kx, (n, d))
true_beta = random.normal(kb, (d,)) * 0.7
true_intercept = 0.75
y = X @ true_beta + true_intercept + random.normal(kn, (n,)) * 0.5

print(f"X: {X.shape}  y: {y.shape}")
print(f"y mean={y.mean():.2f}  std={y.std():.2f}")

## 2. Model

Define once, run under any inference method.

- `@autoreshape` — ensures 1D arrays get a trailing dim added automatically  
- `gaussian_link(mu, y)` — learns `sigma ~ Exp(1)` by default; pass `scale=` or `untransformed_scale=` to fix/learn it

In [ ]:
@autoreshape
def model(x, y=None):
    mu = AdaptiveLayer()("beta", x)          # adaptive prior on weights
    intercept = InterceptLayer()("intercept") # learned intercept
    return gaussian_link(mu + intercept, y)   # sigma learned from data

## 3. Fit with VI (default)

`fit()` handles the guide, ELBO, batching, and LR schedule.
Pass model data as keyword args — arrays are batched, scalars are bound as constants.

In [ ]:
result = fit(
    model,
    y=y,
    num_steps=1000,
    batch_size=256,
    lr=0.01,
    seed=0,
    x=X,
)

plt.plot(result.losses)
plt.xlabel("step")
plt.ylabel("ELBO loss")
plt.title("VI training loss")
plt.show()

## 4. Predict

`result.predict()` draws posterior predictive samples and returns a `Predictions` object
with `.mean`, `.std`, and `.samples`.

In [ ]:
preds = result.predict(x=X, num_samples=500)

rmse = float(jnp.sqrt(jnp.mean((preds.mean - y) ** 2)))
print(f"Posterior predictive RMSE: {rmse:.4f}")

plt.scatter(y, preds.mean, s=8, alpha=0.5)
lims = [float(y.min()), float(y.max())]
plt.plot(lims, lims, "r--", lw=1)
plt.xlabel("True y")
plt.ylabel("Predicted mean")
plt.title("Posterior predictive fit")
plt.show()

## 5. Summarise the posterior

`result.summary()` returns mean, std, and 95% credible intervals for every latent variable.

In [ ]:
summary = result.summary(x=X)

for name, stats in summary.items():
    m = stats["mean"]
    s = stats["std"]
    print(f"{name:40s}  mean={m.mean():.3f}  std={s.mean():.3f}")

## 6. Swap to MCMC — same model, no changes

Set `method="mcmc"`. Use `num_mcmc_samples` / `num_warmup` instead of `num_steps`.

In [ ]:
result_mcmc = fit(
    model,
    y=y,
    method="mcmc",
    num_mcmc_samples=200,
    num_warmup=200,
    seed=0,
    x=X,
)

preds_mcmc = result_mcmc.predict(x=X, num_samples=200)
rmse_mcmc = float(jnp.sqrt(jnp.mean((preds_mcmc.mean - y) ** 2)))
print(f"MCMC posterior predictive RMSE: {rmse_mcmc:.4f}")

## 7. Passing a fixed scale

If you already know the noise level (e.g. from a previous model stage), skip learning sigma:

In [ ]:
@autoreshape
def model_fixed_scale(x, y=None):
    mu = AdaptiveLayer()("beta", x)
    intercept = InterceptLayer()("intercept")
    return gaussian_link(mu + intercept, y, scale=0.5)  # sigma fixed at 0.5


result_fixed = fit(
    model_fixed_scale,
    y=y,
    num_steps=500,
    lr=0.01,
    seed=0,
    x=X,
)

preds_fixed = result_fixed.predict(x=X, num_samples=200)
rmse_fixed = float(jnp.sqrt(jnp.mean((preds_fixed.mean - y) ** 2)))
print(f"Fixed-scale RMSE: {rmse_fixed:.4f}")

## 8. Learned scale from a layer

For heteroskedastic models, learn a per-observation scale via `untransformed_scale=`.
`softplus` is applied internally so gradients stay bounded.

In [ ]:
@autoreshape
def model_learned_scale(x, y=None):
    mu = AdaptiveLayer()("beta", x)
    intercept = InterceptLayer()("intercept")
    log_sigma = AdaptiveLayer()("log_sigma", x)  # raw, unbounded
    return gaussian_link(mu + intercept, y, untransformed_scale=log_sigma)


result_hetero = fit(
    model_learned_scale,
    y=y,
    num_steps=1000,
    lr=0.01,
    seed=0,
    x=X,
)

preds_hetero = result_hetero.predict(x=X, num_samples=200)
rmse_hetero = float(jnp.sqrt(jnp.mean((preds_hetero.mean - y) ** 2)))
print(f"Heteroskedastic model RMSE: {rmse_hetero:.4f}")